# 🧮 02 · 矩阵分解与隐语义模型

> 协同过滤的进阶：把 user-item 评分矩阵分解成**用户隐向量 × 物品隐向量**，用隐向量点积预测评分。
> 本节手写 **Funk-SVD（SGD 矩阵分解）** 并对比 SVD 截断，最后可视化 embedding 语义。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 隐语义模型的思想？ | 矩阵分解 = 发现潜在因子 |
| 2 | 怎么求解？ | SGD / 交替最小二乘 ALS |
| 3 | 和 SVD 有什么关系？ | 截断 SVD 是闭式解 |
| 4 | 隐向量有什么意义？ | embedding 可视化 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造数据（含隐性反馈的评分）

In [2]:
n_u, n_i = 40, 30
true_u = rng.normal(0, 1, (n_u, 4))
true_v = rng.normal(0, 1, (n_i, 4))
R = 3.0 + true_u @ true_v.T + rng.normal(0, 0.3, (n_u, n_i))  # 评分尺度 1~5 左右
# 随机遮盖 30% 作为缺失
mask = rng.uniform(0, 1, R.shape) > 0.3
R = np.where(mask, R, 0.0)
print('评分矩阵形状:', R.shape, '| 观测比例:', (R > 0).mean().round(3))
assert 0.6 < (R > 0).mean() < 0.8
print('✅ 数据就绪（含缺失）')

评分矩阵形状: (40, 30) | 观测比例: 0.67
✅ 数据就绪（含缺失）


## 2️⃣ Funk-SVD：手写 SGD 分解

In [3]:
def funk_svd(R, k=4, lr=0.02, reg=0.05, epochs=150, seed=7):
    rng2 = np.random.default_rng(seed)
    n_u, n_i = R.shape
    U = rng2.normal(0, 0.1, (n_u, k))
    V = rng2.normal(0, 0.1, (n_i, k))
    bu = np.zeros(n_u)   # 用户偏差
    bi = np.zeros(n_i)   # 物品偏差
    mu = R[R > 0].mean()
    observed = np.argwhere(R > 0)
    history = []
    for ep in range(epochs):
        rng2.shuffle(observed)
        for u, i in observed:
            pred = mu + bu[u] + bi[i] + U[u] @ V[i]
            err = R[u, i] - pred
            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])
            U[u] += lr * (err * V[i] - reg * U[u])
            V[i] += lr * (err * U[u] - reg * V[i])
        # 每 30 轮记录 RMSE
        if ep % 30 == 0:
            preds = mu + bu[:, None] + bi[None, :] + U @ V.T
            rmse = np.sqrt(((preds - R)[R > 0] ** 2).mean())
            history.append((ep, rmse))
    return U, V, bu, bi, mu, history

U, V, bu, bi, mu, hist = funk_svd(R)
for ep, rmse in hist:
    print(f'epoch {ep}: RMSE = {rmse:.4f}')
final_rmse = hist[-1][1]
assert final_rmse < 0.5
print('✅ 训练收敛，RMSE 下降')

epoch 0: RMSE = 1.4934
epoch 30: RMSE = 0.5154
epoch 60: RMSE = 0.2643
epoch 90: RMSE = 0.2609
epoch 120: RMSE = 0.2614
✅ 训练收敛，RMSE 下降


## 3️⃣ 预测与评估

In [4]:
pred_full = mu + bu[:, None] + bi[None, :] + U @ V.T

# 训练集内评估
train_mask = R > 0
rmse_train = np.sqrt(((pred_full - R)[train_mask] ** 2).mean())
# 模拟留出 10% 观测做验证（这里简化：随机抽 10% 观测点算误差）
test_mask = (rng.uniform(0, 1, R.shape) < 0.1) & train_mask
rmse_test = np.sqrt(((pred_full - R)[test_mask] ** 2).mean()) if test_mask.sum() > 0 else 0
print(f'训练 RMSE = {rmse_train:.4f}')
print(f'抽样验证 RMSE = {rmse_test:.4f}')
assert rmse_test < 0.6
print('✅ 预测误差可控')

训练 RMSE = 0.2610
抽样验证 RMSE = 0.2521
✅ 预测误差可控


## 4️⃣ 与截断 SVD 对比（闭式解）

In [5]:
# 用已观测值补 0 后做 SVD（真实工业会用加权/填充均值，这里演示思路）
R_fill = np.where(R > 0, R, R[R > 0].mean())
U_svd, S_svd, Vt_svd = np.linalg.svd(R_fill, full_matrices=False)
k = 4
U_hat = U_svd[:, :k] @ np.diag(np.sqrt(S_svd[:k]))
V_hat = Vt_svd[:k, :].T @ np.diag(np.sqrt(S_svd[:k]))
pred_svd = U_hat @ V_hat.T
rmse_svd = np.sqrt(((pred_svd - R)[R > 0] ** 2).mean())
print(f'截断 SVD(k=4) 观测点 RMSE = {rmse_svd:.4f}')
print('➡️ SVD 是全局最优闭式解；Funk-SVD 只优化观测点，适合稀疏数据且可加偏差/时间项')

截断 SVD(k=4) 观测点 RMSE = 0.7953
➡️ SVD 是全局最优闭式解；Funk-SVD 只优化观测点，适合稀疏数据且可加偏差/时间项


## 5️⃣ Embedding 可视化：隐向量里有什么语义

In [6]:
# 物品隐向量投影到 2D（PCA）
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
item_2d = pca.fit_transform(V)

plt.figure(figsize=(7, 5))
plt.scatter(item_2d[:, 0], item_2d[:, 1], s=60, c=np.arange(n_i), cmap='viridis')
for j in range(n_i):
    plt.annotate(f'I{j+1}', (item_2d[j, 0], item_2d[j, 1]), fontsize=9)
plt.title('物品隐向量 2D 投影（语义相近的物品距离近）')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()
print('➡️ 隐向量本质是低维嵌入：相似物品在隐空间相邻，可做向量召回、聚类、解释')

➡️ 隐向量本质是低维嵌入：相似物品在隐空间相邻，可做向量召回、聚类、解释


C:\Users\24260\AppData\Local\Temp\ipykernel_47652\1903069265.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6️⃣ 面试高频追问（速答）

In [7]:
print('''1) 为什么用低秩分解？  → 隐因子压缩信息、泛化、抗稀疏；
2) SGD vs ALS 怎么选？   → 稀疏/可扩展 ALS 更快（交替最小二乘闭式更新），
   SGD 灵活可加正则/偏差/时间衰减；
3) 隐向量维度 k 怎么定？ → 交叉验证；一般几十~几百；
4) 和 DeepFM 关系？     → DeepFM 的 FM 项 = 特征 embedding 的点积，
   矩阵分解是「只有 id 特征」的特例；
5) 冷启动怎么办？       → 内容特征/热启动模型（DNN 用内容 embedding 替代 id）。
''')

1) 为什么用低秩分解？  → 隐因子压缩信息、泛化、抗稀疏；
2) SGD vs ALS 怎么选？   → 稀疏/可扩展 ALS 更快（交替最小二乘闭式更新），
   SGD 灵活可加正则/偏差/时间衰减；
3) 隐向量维度 k 怎么定？ → 交叉验证；一般几十~几百；
4) 和 DeepFM 关系？     → DeepFM 的 FM 项 = 特征 embedding 的点积，
   矩阵分解是「只有 id 特征」的特例；
5) 冷启动怎么办？       → 内容特征/热启动模型（DNN 用内容 embedding 替代 id）。



## 📝 自测清单

- [ ] 能手写 Funk-SVD（偏差+正则+SGD）并解释每步
- [ ] 能解释截断 SVD 与 Funk-SVD 的差异
- [ ] 能讲清隐向量/embedding 的语义与用途
- [ ] 能回答 k 怎么选、SGD vs ALS 怎么选
- [ ] 能说明矩阵分解与 FM/DeepFM 的关系